In [ ]:
CITY_COORDINATES = {
    "Cairo": (30.0444, 31.2357),
    "Alexandria": (31.2001, 29.9187),
    "Giza": (30.0131, 31.2089),
    "Luxor": (25.6872, 32.6396),
    "Aswan": (24.0889, 32.8998),
    "Port Said": (31.2653, 32.3019),
}

In [ ]:
import requests

lat, lon = CITY_COORDINATES["Cairo"]
response = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params={"latitude": lat, "longitude": lon, "current_weather": "true"}
)

print("Status code:", response.status_code)
print("Raw text:", response.text)

Status code: 200
Raw text: {"latitude":30.0625,"longitude":31.25,"generationtime_ms":0.7543563842773438,"utc_offset_seconds":0,"timezone":"GMT","timezone_abbreviation":"GMT","elevation":22.0,"current_weather_units":{"time":"iso8601","interval":"seconds","temperature":"°C","windspeed":"km/h","winddirection":"°","is_day":"","weathercode":"wmo code"},"current_weather":{"time":"2026-07-10T13:00","interval":900,"temperature":36.2,"windspeed":9.7,"winddirection":345,"is_day":1,"weathercode":1}}


In [ ]:
data = response.json()
print("Type after .json():", type(data))
data

Type after .json(): <class 'dict'>


{'latitude': 30.0625,
 'longitude': 31.25,
 'generationtime_ms': 0.7543563842773438,
 'utc_offset_seconds': 0,
 'timezone': 'GMT',
 'timezone_abbreviation': 'GMT',
 'elevation': 22.0,
 'current_weather_units': {'time': 'iso8601',
  'interval': 'seconds',
  'temperature': '°C',
  'windspeed': 'km/h',
  'winddirection': '°',
  'is_day': '',
  'weathercode': 'wmo code'},
 'current_weather': {'time': '2026-07-10T13:00',
  'interval': 900,
  'temperature': 36.2,
  'windspeed': 9.7,
  'winddirection': 345,
  'is_day': 1,
  'weathercode': 1}}

In [ ]:
current = data["current_weather"]
print("Temperature (C):", current["temperature"])
print("Wind speed (km/h):", current["windspeed"])
print("Observation time:", current["time"])

Temperature (C): 36.2
Wind speed (km/h): 9.7
Observation time: 2026-07-10T13:00


In [ ]:
import time

weather_records = []

for city, (lat, lon) in CITY_COORDINATES.items():
    response = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={"latitude": lat, "longitude": lon, "current_weather": "true"}
    )
    if response.status_code == 200:
        current = response.json()["current_weather"]
        weather_records.append({
            "city": city,
            "temperature_c": current["temperature"],
            "windspeed_kmh": current["windspeed"],
            "observed_at": current["time"]
        })
    time.sleep(0.3)  # small pause to be polite to the free API

weather_records

[{'city': 'Cairo',
  'temperature_c': 36.2,
  'windspeed_kmh': 9.7,
  'observed_at': '2026-07-10T13:00'},
 {'city': 'Alexandria',
  'temperature_c': 29.1,
  'windspeed_kmh': 13.8,
  'observed_at': '2026-07-10T13:00'},
 {'city': 'Giza',
  'temperature_c': 36.3,
  'windspeed_kmh': 10.6,
  'observed_at': '2026-07-10T13:00'},
 {'city': 'Luxor',
  'temperature_c': 39.7,
  'windspeed_kmh': 16.9,
  'observed_at': '2026-07-10T13:00'},
 {'city': 'Aswan',
  'temperature_c': 38.9,
  'windspeed_kmh': 16.9,
  'observed_at': '2026-07-10T13:00'},
 {'city': 'Port Said',
  'temperature_c': 30.0,
  'windspeed_kmh': 18.0,
  'observed_at': '2026-07-10T13:00'}]

In [ ]:
import pandas as pd

weather_df = pd.DataFrame(weather_records)
weather_df

,city,temperature_c,windspeed_kmh,observed_at
0,Cairo,36.2,9.7,2026-07-10T13:00
1,Alexandria,29.1,13.8,2026-07-10T13:00
2,Giza,36.3,10.6,2026-07-10T13:00
3,Luxor,39.7,16.9,2026-07-10T13:00
4,Aswan,38.9,16.9,2026-07-10T13:00
5,Port Said,30.0,18.0,2026-07-10T13:00


In [ ]:
HEADERS = {"User-Agent": "DigitalEgyptCubsEduBot/1.0 (educational classroom project)"}

page = requests.get(
    "https://en.wikipedia.org/wiki/List_of_cities_and_towns_in_Egypt",
    headers=HEADERS
)
print("Status code:", page.status_code)

Status code: 200


In [ ]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(page.text, "html.parser")

print("Page title:", soup.find("h1").text)

tables = soup.find_all("table", class_="wikitable")
print("Number of 'wikitable' tables found:", len(tables))

first_table_headers = [th.text.strip() for th in tables[0].find_all("th")]
print("Columns in the first table:", first_table_headers)

Page title: List of cities and towns in Egypt
Number of 'wikitable' tables found: 1
Columns in the first table: ['Name', 'Arabic', 'Governorate', 'Area code', 'Population (2023 estimate)[11]', 'Photo']


In [ ]:
from io import StringIO

all_tables = pd.read_html(StringIO(page.text))
print("Number of tables pandas found:", len(all_tables))

# Find the table that has a population-related column — that's the one we want
city_table = None
for t in all_tables:
    col_names = [str(c).lower() for c in t.columns]
    if any("population" in c for c in col_names) and any("city" in c or "name" in c for c in col_names):
        city_table = t
        break

city_table.head()

Number of tables pandas found: 15


,Name,Arabic,Governorate,Area code,Population (2023 estimate)[11],Photo
0,Cairo*,القاهرة,Cairo,(+20) 2,10000000,NaN
1,Alexandria,الاسكندرية,Alexandria,(+20) 3,5362517,NaN
2,Giza*,الجيزة,Giza,(+20) 2,4458135,NaN
3,Shubra El Kheima*,شبرا الخيمة,Qalyubia,(+20) 2,1275700,NaN
4,Port Said,بور سعيد,Port Said,(+20) 60,791749,NaN


In [ ]:
print(list(city_table.columns))

['Name', 'Arabic', 'Governorate', 'Area code', 'Population (2023 estimate)[11]', 'Photo']


In [ ]:
CITY_COL = "Name"
GOVERNORATE_COL = [c for c in city_table.columns if "governorate" in str(c).lower()][0]
POPULATION_COL = [c for c in city_table.columns if "population" in str(c).lower()][0]

target_cities = list(CITY_COORDINATES.keys())

# Clean the scraped names: remove the asterisk and any extra spaces
city_table["clean_name"] = city_table[CITY_COL].str.replace("*", "", regex=False).str.strip()

# Keep only rows whose cleaned name is EXACTLY one of our 6 target cities
station_df = city_table[city_table["clean_name"].isin(target_cities)][
    ["clean_name", GOVERNORATE_COL, POPULATION_COL]
].copy()

station_df.columns = ["city", "governorate", "population"]
station_df = station_df.reset_index(drop=True)
station_df

,city,governorate,population
0,Cairo,Cairo,10000000
1,Alexandria,Alexandria,5362517
2,Giza,Giza,4458135
3,Port Said,Port Said,791749
4,Aswan,Aswan,401890
5,Luxor,Luxor,284952


In [ ]:
combined_df = pd.merge(weather_df, station_df, on="city")
combined_df

,city,temperature_c,windspeed_kmh,observed_at,governorate,population
0,Cairo,36.2,9.7,2026-07-10T13:00,Cairo,10000000
1,Alexandria,29.1,13.8,2026-07-10T13:00,Alexandria,5362517
2,Giza,36.3,10.6,2026-07-10T13:00,Giza,4458135
3,Luxor,39.7,16.9,2026-07-10T13:00,Luxor,284952
4,Aswan,38.9,16.9,2026-07-10T13:00,Aswan,401890
5,Port Said,30.0,18.0,2026-07-10T13:00,Port Said,791749


In [ ]:
combined_df.to_csv("egypt_weather_collected.csv", index=False)
print("Saved to egypt_weather_collected.csv")

Saved to egypt_weather_collected.csv


In [ ]:
reloaded_df = pd.read_csv("egypt_weather_collected.csv")
reloaded_df.head()

,city,temperature_c,windspeed_kmh,observed_at,governorate,population
0,Cairo,36.2,9.7,2026-07-10T13:00,Cairo,10000000
1,Alexandria,29.1,13.8,2026-07-10T13:00,Alexandria,5362517
2,Giza,36.3,10.6,2026-07-10T13:00,Giza,4458135
3,Luxor,39.7,16.9,2026-07-10T13:00,Luxor,284952
4,Aswan,38.9,16.9,2026-07-10T13:00,Aswan,401890


In [ ]:
reloaded_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   city           6 non-null      object 
 1   temperature_c  6 non-null      float64
 2   windspeed_kmh  6 non-null      float64
 3   observed_at    6 non-null      object 
 4   governorate    6 non-null      object 
 5   population     6 non-null      int64  
dtypes: float64(2), int64(1), object(3)
memory usage: 420.0+ bytes


In [ ]:
reloaded_df.describe()

,temperature_c,windspeed_kmh,population
count,6.000000,6.000000,6.000000e+00
mean,35.033333,14.316667,3.549874e+06
std,4.477350,3.530109,3.844256e+06
min,29.100000,9.700000,2.849520e+05
25%,31.550000,11.400000,4.993548e+05
50%,36.250000,15.350000,2.624942e+06
75%,38.250000,16.900000,5.136422e+06
max,39.700000,18.000000,1.000000e+07


In [ ]:
print("Number of rows:", reloaded_df.shape[0])
print("Number of columns:", reloaded_df.shape[1])
print("Missing values per column:")
print(reloaded_df.isnull().sum())

Number of rows: 6
Number of columns: 6
Missing values per column:
city             0
temperature_c    0
windspeed_kmh    0
observed_at      0
governorate      0
population       0
dtype: int64


### Recap

In this project we:
1. Called a real **API** (Open-Meteo) to collect live weather readings for 6 cities
2. **Scraped** a real Wikipedia page to collect city facts
3. Combined both sources into a **pandas DataFrame**
4. Saved the result as a **CSV** file
5. Reloaded the CSV and explored it with `.head()`, `.info()`, and `.describe()`

This is a complete, simple **data collection pipeline** using real, open, freely available sources — the same pattern used in real data science projects.